# 入力の途中で続きを予測する（予測入力）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_japanese_input/predictive_input.ipynb)

NLP クックブック 26-08 ／ 日本語入力（かな漢字変換）

数文字入力しただけで、入力しようとしている単語や、次に来そうな単語を候補として出したい（スマートフォンの入力のように）。

In [1]:
!pip install -q fugashi unidic-lite jaconv marisa-trie

In [2]:
import io, re, urllib.request, zipfile
from collections import Counter, defaultdict
import jaconv
import marisa_trie
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
word_count, next_word, phrase_count = Counter(), defaultdict(Counter), Counter()
for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"]:
    for s in re.split(r"(?<=。)|\n", aozora(base + f)):
        words = [(w.surface, jaconv.kata2hira(w.feature.kana or w.surface), w.feature.pos1) for w in tagger(s.strip())]
        for i, (surface, reading, pos) in enumerate(words):
            if pos not in ("補助記号", "空白"):
                word_count[reading, surface] += 1
            if i + 1 < len(words):
                next_word[surface][words[i + 1][0]] += 1
            if i + 2 < len(words) and pos in ("名詞", "動詞", "形容詞"):     # 内容語から始まる 3 語の言い回しも候補にする
                phrase_count["".join(r for _, r, _ in words[i:i + 3]), "".join(s for s, _, _ in words[i:i + 3])] += 1

# 読みの前方一致で引けるように、「読み\t表記」をキーにしたトライ木を作る
entries = {f"{r}\t{s}": c for (r, s), c in list(word_count.items()) + [(k, c) for k, c in phrase_count.items() if c >= 5]}
trie = marisa_trie.Trie(entries.keys())
print(len(entries), "件")

20258 件


In [3]:
def predict(prefix, k=6):
    """入力中の読みで前方一致する語句を、頻度の高い順に出す"""
    hits = [(entries[key], key.split("\t")) for key in trie.keys(prefix)]
    hits = [(c, r, s) for c, (r, s) in hits if r != prefix or s != prefix]
    return [s for c, r, s in sorted(hits, reverse=True)[:k]]

for prefix in ["わ", "わが", "せんせ", "にんげ", "おも", "しか", "かんが"]:
    print(f"{prefix} → {predict(prefix)}")

わ → ['私', '吾輩', '訳', '分ら', '笑っ', '笑い']
わが → ['吾輩', '我儘', 'わが家', '我', '吾が', '我が']
せんせ → ['先生', '先生は、', '先生の宅', '先生の言葉', '先生の顔', '先生は私']
にんげ → ['人間', '人間とし', '人間という', '人間には', '人間らしくない', '人間とは']
おも → ['思っ', '思う', '思わ', '思い', '面白い', '思った。']
しか → ['しかし', '仕方', '仕方がない', 'しかも', 'しかる', '資格']
かんが → ['考え', '考える', '考えてい', '考えている', '考', '考えた。']


In [4]:
def predict_next(previous, k=6):
    """直前に確定した単語の次に来やすい単語を出す (何も入力しなくても候補が出る)"""
    return [w for w, _ in next_word[previous].most_common(k)]

for w in ["吾輩", "先生", "東京", "しかし", "考え"]:
    print(f"{w} の次 → {predict_next(w)}")

# 入力の手間がどれだけ減るか: 候補の上位 3 件に目的の語が出るまでに、何文字打つ必要があるか
def keystrokes(reading, surface, top=3):
    for n in range(1, len(reading) + 1):
        if surface in predict(reading[:n], top):
            return n
    return len(reading)

sample = [(r, s) for (r, s), c in word_count.most_common(2000) if len(r) >= 3][:300]
saved = sum(len(r) - keystrokes(r, s) for r, s in sample) / sum(len(r) for r, _ in sample)
print(f"頻出 300 語で、打鍵数を {saved:.0%} 減らせる")

吾輩 の次 → ['は', 'の', 'が', 'を', 'も', 'に']
先生 の次 → ['は', 'の', 'が', 'と', 'に', 'を']
東京 の次 → ['へ', 'を', 'の', 'に', 'から', 'で']
しかし の次 → ['私', 'その', 'それ', '先生', 'この', '奥']
考え の次 → ['て', 'た', 'まし', 'が', 'で', 'を']
頻出 300 語で、打鍵数を 45% 減らせる


- 予測入力は「読みの前方一致 + 頻度順」が基本。トライ木なら、入力中の文字列で始まる候補を高速に列挙できる。
- 単語だけでなく、よく使う言い回し（複数の単語のまとまり）も候補にすると、打鍵数がさらに減る。
- 直前の単語の次に来やすい単語を出すと、何も打たなくても候補が出る。ここでは bigram を使ったが、ニューラル言語モデルを使うと、より長い文脈を踏まえた予測ができる。
- 利用者ごとの履歴（最近入力した語を上位に出す）が、実際の使い勝手を大きく左右する。この章の学習のレシピを参照。
- 予測候補に個人情報や不適切な語が出ないようにする配慮も必要。履歴の学習では、パスワード欄などの入力を記録しない。